In [ ]:
import sys
print("Jupyter Python:", sys.executable)
print("Python version:", sys.version)

Jupyter Python: /home/karpagam/conda_envs/RT_tf/bin/python
Python version: 3.10.20 (main, Jun 11 2026, 15:17:37) [GCC 14.3.0]


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import re
import gc
import json
import glob
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import nibabel as nib
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.stats import wilcoxon

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

BASE_PATH = Path(
    "/media/karpagam/"
    "5b08ffff-26a8-4032-b58c-ed102cf002e2/"
    "aarthi/MapASD"
)

ABIDE2_PATH = BASE_PATH / "ABIDE2"
CSV_PATH = BASE_PATH / "ABIDEII_Composite_Phenotypic.csv"

OUTPUT_DIR = BASE_PATH / "mapasdnet_accuracy_v2_results"
CACHE_DIR = OUTPUT_DIR / "slice_cache"
MODEL_DIR = OUTPUT_DIR / "models"
RESULTS_DIR = OUTPUT_DIR / "results"
SPLITS_DIR = OUTPUT_DIR / "splits"
LOGS_DIR = OUTPUT_DIR / "logs"
FIGURES_DIR = OUTPUT_DIR / "figures"
ABLATION_DIR = OUTPUT_DIR / "ablation"
CROSS_VALIDATION_DIR = OUTPUT_DIR / "cross_validation"
LOSO_DIR = OUTPUT_DIR / "loso"

for directory in [
    OUTPUT_DIR,
    CACHE_DIR,
    MODEL_DIR,
    RESULTS_DIR,
    SPLITS_DIR,
    LOGS_DIR,
    FIGURES_DIR,
    ABLATION_DIR,
    CROSS_VALIDATION_DIR,
    LOSO_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

DATA_FOLDERS = [
    ABIDE2_PATH / "ABIDEII-BNI_1",
    ABIDE2_PATH / "ABIDEII-EMC_1",
    ABIDE2_PATH / "ABIDEII-ETH_1",
    ABIDE2_PATH / "ABIDEII-GU_1",
    ABIDE2_PATH / "ABIDEII-IP_1",
    ABIDE2_PATH / "ABIDEII-IU_1",
    ABIDE2_PATH / "ABIDEII-KKI_1_29273_29322A",
    ABIDE2_PATH / "ABIDEII-KKI_1_29323_29372B",
    ABIDE2_PATH / "ABIDEII-KKI_1_29373_29423C",
    ABIDE2_PATH / "ABIDEII-KKI_1_29424_29485D",
    ABIDE2_PATH / "ABIDEII-KUL_3",
    ABIDE2_PATH / "ABIDEII-NYU_1",
    ABIDE2_PATH / "ABIDEII-NYU_2",
    ABIDE2_PATH / "ABIDEII-OHSU_1",
    ABIDE2_PATH / "ABIDEII-ONRC_2_part1",
    ABIDE2_PATH / "ABIDEII-ONRC_2_part2",
    ABIDE2_PATH / "ABIDEII-ONRC_2_part3",
    ABIDE2_PATH / "ABIDEII-ONRC_2_part4",
    ABIDE2_PATH / "ABIDEII-SDSU_1",
    ABIDE2_PATH / "ABIDEII-SU_2",
    ABIDE2_PATH / "ABIDEII-TCD_1",
    ABIDE2_PATH / "ABIDEII-UCD_1",
]

# MRI settings
IMG_SIZE = 160
N_SLICES = 25
SLICE_STEP = 3
LOWER_INTENSITY_PERCENTILE = 1.0
UPPER_INTENSITY_PERCENTILE = 99.0
TARGET_VOXEL_MM = 2.0
PREPROCESSING_VERSION = "v3_canonical_2mm_foreground_crop"

# Training settings
BATCH_SIZE = 8
EPOCHS = 80
PATIENCE = 12
LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 1e-6
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE = 5
WEIGHT_DECAY = 1e-5

# Revised regularization settings
LSTM_DROPOUT = 0.15
IMAGE_DROPOUT = 0.15
METADATA_DROPOUT = 0.10
SHARED_DROPOUT = 0.20
SITE_DROPOUT = 0.20

# Class weighting is unnecessary for the mildly imbalanced primary cohort.
USE_CLASS_WEIGHTS = False

# Target-model ensemble; seeds are fixed before test evaluation.
ENSEMBLE_SEEDS = [42, 52, 62]

# Adversarial settings
SITE_LOSS_WEIGHT = 0.03
GRL_MAX_COEFFICIENT = 0.05
GRL_WARMUP_EPOCHS = 10

# Evaluation
N_FOLDS = 5
THRESHOLD_MIN = 0.10
THRESHOLD_MAX = 0.90
THRESHOLD_STEP = 0.01
THRESHOLD_OBJECTIVE = "accuracy"

# Cache
USE_CACHE = True
REBUILD_CACHE = False

# CSV columns
SUBJECT_COLUMN = "SUB_ID"
LABEL_COLUMN = "DX_GROUP"
SITE_COLUMN = "SITE_ID"

NUMERIC_METADATA_COLUMNS = [
    "AGE_AT_SCAN",
    "FIQ",
    "VIQ",
    "PIQ",
]

CATEGORICAL_METADATA_COLUMNS = [
    "SEX",
]


I0000 00:00:1785814999.765302   17715 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785815000.008231   17715 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785815001.975033   17715 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Core data, preprocessing, model, CV, and LOSO functions

In [ ]:
# ============================================================
# SHARED EXPERIMENT DEFAULTS
# ============================================================
CV_EXPERIMENTS = [
    "cnn_only", "lstm_only", "bilstm_only", "cnn_lstm",
    "cnn_bilstm", "cnn_bilstm_attention",
    "cnn_bilstm_attention_metadata", "cnn_bilstm_attention_metadata_adversarial",
]
LOSO_EXPERIMENT = "cnn_bilstm_attention_metadata_adversarial"

# ============================================================
# REPRODUCIBILITY AND DEVICE SETUP
# ============================================================

def set_global_seed(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def configure_gpu_memory_growth() -> None:
    gpus = tf.config.list_physical_devices("GPU")

    if not gpus:
        print("TensorFlow GPU not detected. Training will use CPU.")
        return

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as exc:
            print(f"Could not configure memory growth: {exc}")

    print("Detected TensorFlow GPUs:")
    for gpu in gpus:
        print(f"  - {gpu}")


set_global_seed(SEED)
configure_gpu_memory_growth()


# ============================================================
# PATH VALIDATION
# ============================================================

def validate_paths() -> List[Path]:
    print("=" * 80)
    print("MAPASDNET PATH VALIDATION")
    print("=" * 80)
    print(f"Base path      : {BASE_PATH}")
    print(f"ABIDE-II path  : {ABIDE2_PATH}")
    print(f"Phenotypic CSV : {CSV_PATH}")
    print(f"Output path    : {OUTPUT_DIR}")

    if not BASE_PATH.exists():
        raise FileNotFoundError(f"BASE_PATH does not exist:\n{BASE_PATH}")

    if not ABIDE2_PATH.exists():
        raise FileNotFoundError(f"ABIDE2_PATH does not exist:\n{ABIDE2_PATH}")

    if not CSV_PATH.exists():
        raise FileNotFoundError(f"CSV_PATH does not exist:\n{CSV_PATH}")

    existing = [folder for folder in DATA_FOLDERS if folder.exists()]
    missing = [folder for folder in DATA_FOLDERS if not folder.exists()]

    print(f"\nConfigured folders: {len(DATA_FOLDERS)}")
    print(f"Existing folders  : {len(existing)}")
    print(f"Missing folders   : {len(missing)}")

    if missing:
        print("\nMissing folders:")
        for folder in missing:
            print(f"  - {folder}")

    if not existing:
        raise RuntimeError("No configured ABIDE-II data folders were found.")

    return existing


# ============================================================
# DATASET DISCOVERY AND SUBJECT MATCHING
# ============================================================

def extract_subject_id(path: str) -> Optional[int]:
    """
    Extract the subject ID from a path.

    First checks purely numeric path components, then numeric groups
    of length five or more in the filename/path.
    """
    path_obj = Path(path)

    for part in reversed(path_obj.parts):
        if part.isdigit():
            return int(part)

    matches = re.findall(r"(?<!\d)(\d{5,})(?!\d)", str(path_obj))
    if matches:
        return int(matches[-1])

    return None


def is_probable_t1_file(path: str) -> bool:
    """
    Retain probable anatomical T1 files and reject common derivatives.
    Adjust these patterns if your local filenames use different terms.
    """
    name = Path(path).name.lower()
    full_path = str(path).lower()

    excluded = [
        "mask",
        "seg",
        "label",
        "parc",
        "template",
        "mean",
        "bold",
        "func",
        "rest",
        "fmri",
    ]

    if any(term in name for term in excluded):
        return False

    accepted = ["anat", "t1", "mprage", "mpr"]

    return any(term in name or term in full_path for term in accepted)


def infer_site_from_path(path: str) -> str:
    for part in Path(path).parts:
        if part.startswith("ABIDEII-"):
            return part
    return "UNKNOWN"


def find_candidate_mri_files(data_folders: List[Path]) -> List[str]:
    all_files: List[str] = []

    for folder in data_folders:
        matches = glob.glob(
            str(folder / "**" / "*.nii.gz"),
            recursive=True,
        )
        all_files.extend(matches)

    print(f"\nRaw .nii.gz files found: {len(all_files)}")

    probable_t1 = [path for path in all_files if is_probable_t1_file(path)]
    print(f"Probable T1 files       : {len(probable_t1)}")

    if not probable_t1:
        print(
            "\nWarning: no files passed the T1 filename filter. "
            "Falling back to all NIfTI files. Review the selected files carefully."
        )
        probable_t1 = all_files

    return sorted(probable_t1)


def choose_one_file_per_subject(records: pd.DataFrame) -> pd.DataFrame:
    """
    Deterministically choose one probable T1 file per participant.

    Preference order:
    1. Filename containing 'mprage'
    2. Filename containing 't1'
    3. Filename containing 'anat'
    4. Shorter path
    5. Alphabetical path
    """

    def priority(path: str) -> Tuple[int, int, str]:
        name = Path(path).name.lower()
        if "mprage" in name:
            rank = 0
        elif "t1" in name:
            rank = 1
        elif "anat" in name:
            rank = 2
        else:
            rank = 3
        return rank, len(path), path

    records = records.copy()
    priorities = records["file_path"].map(priority)

    records["_rank"] = priorities.map(lambda item: item[0])
    records["_length"] = priorities.map(lambda item: item[1])

    records = (
        records.sort_values(
            ["subject_id", "_rank", "_length", "file_path"]
        )
        .drop_duplicates(subset=["subject_id"], keep="first")
        .drop(columns=["_rank", "_length"])
        .reset_index(drop=True)
    )

    return records


def create_subject_table(
    csv_path: Path,
    data_folders: List[Path],
) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding="latin1")
    # ABIDE-II contains at least one heading with trailing whitespace
    # (for example 'AGE_AT_SCAN '). Normalize headings before lookup.
    df.columns = df.columns.astype(str).str.strip()

    print("\nCSV columns:")
    print(df.columns.tolist())

    required = {SUBJECT_COLUMN, LABEL_COLUMN}
    missing_required = required.difference(df.columns)

    if missing_required:
        raise ValueError(
            f"Missing required phenotypic columns: {sorted(missing_required)}"
        )

    candidate_files = find_candidate_mri_files(data_folders)

    rows: List[Dict] = []

    for path in candidate_files:
        subject_id = extract_subject_id(path)

        if subject_id is None:
            continue

        matches = df[df[SUBJECT_COLUMN] == subject_id]

        if matches.empty:
            continue

        phenotype = matches.iloc[0]

        try:
            diagnosis_value = int(phenotype[LABEL_COLUMN])
        except (TypeError, ValueError):
            continue

        if diagnosis_value not in (1, 2):
            continue

        label = 1 if diagnosis_value == 1 else 0

        if SITE_COLUMN in df.columns and pd.notna(phenotype[SITE_COLUMN]):
            site = str(phenotype[SITE_COLUMN])
        else:
            site = infer_site_from_path(path)

        record = {
            "subject_id": int(subject_id),
            "file_path": str(path),
            "label": int(label),
            "site": site,
        }

        for column in NUMERIC_METADATA_COLUMNS:
            record[column] = (
                phenotype[column] if column in df.columns else np.nan
            )

        for column in CATEGORICAL_METADATA_COLUMNS:
            record[column] = (
                phenotype[column] if column in df.columns else np.nan
            )

        rows.append(record)

    records = pd.DataFrame(rows)

    if records.empty:
        raise RuntimeError(
            "No MRI files matched the phenotypic CSV. "
            "Check subject-ID extraction and directory structure."
        )

    before = len(records)
    records = choose_one_file_per_subject(records)
    after = len(records)

    print(f"\nMatched MRI records before deduplication: {before}")
    print(f"Unique matched subjects              : {after}")
    print(f"ASD subjects                         : {(records['label'] == 1).sum()}")
    print(f"Control subjects                     : {(records['label'] == 0).sum()}")
    print(f"Acquisition sites                    : {records['site'].nunique()}")

    print("\nSite × diagnosis table:")
    site_table = pd.crosstab(
        records["site"],
        records["label"],
        margins=True,
    )
    print(site_table)

    records.to_csv(
        RESULTS_DIR / "matched_subjects.csv",
        index=False,
    )

    site_table.to_csv(
        RESULTS_DIR / "site_diagnosis_summary.csv"
    )

    print("\nFirst matched examples:")
    print(
        records[["subject_id", "site", "label", "file_path"]]
        .head(10)
        .to_string(index=False)
    )

    return records


# ============================================================
# MULTI-SLICE MRI PREPROCESSING
# ============================================================

def robust_minmax_normalize(volume: np.ndarray) -> np.ndarray:
    valid = volume[np.isfinite(volume) & (volume != 0)]

    if valid.size == 0:
        return np.zeros_like(volume, dtype=np.float32)

    low, high = np.percentile(
        valid,
        [LOWER_INTENSITY_PERCENTILE, UPPER_INTENSITY_PERCENTILE],
    )

    if high <= low:
        return np.zeros_like(volume, dtype=np.float32)

    clipped = np.clip(volume, low, high)
    normalized = (clipped - low) / (high - low + 1e-8)

    return normalized.astype(np.float32)


def choose_slice_indices(
    depth: int,
    n_slices: int = N_SLICES,
    slice_step: int = SLICE_STEP,
) -> np.ndarray:
    if n_slices < 1:
        raise ValueError("N_SLICES must be at least 1.")

    centre = depth // 2
    start_offset = -(n_slices // 2) * slice_step
    offsets = start_offset + np.arange(n_slices) * slice_step

    indices = centre + offsets
    indices = np.clip(indices, 0, depth - 1)

    return indices.astype(np.int32)


def load_multi_slice_volume(nii_path: str) -> np.ndarray:
    """Load a 3-D T1, standardize physical spacing, crop padding, and sample slices.

    This standardizes orientation, voxel spacing, and empty field of view. It is
    not a replacement for upstream skull stripping or MNI registration.
    """
    from nibabel.processing import resample_to_output

    image = nib.as_closest_canonical(nib.load(nii_path))
    if len(image.shape) != 3:
        raise ValueError(
            f"Expected a 3D structural MRI, received shape {image.shape}: {nii_path}"
        )

    image = resample_to_output(
        image,
        voxel_sizes=(TARGET_VOXEL_MM,) * 3,
        order=1,
        mode="constant",
        cval=0.0,
    )
    volume = np.asarray(image.get_fdata(dtype=np.float32))
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)

    nonzero = np.abs(volume[np.abs(volume) > 1e-6])
    if nonzero.size == 0:
        raise ValueError(f"Empty structural MRI: {nii_path}")

    foreground_threshold = max(float(np.percentile(nonzero, 5)) * 0.20, 1e-6)
    coordinates = np.argwhere(np.abs(volume) > foreground_threshold)
    lower = coordinates.min(axis=0)
    upper = coordinates.max(axis=0) + 1
    margin = np.maximum(((upper - lower) * 0.05).astype(int), 2)
    lower = np.maximum(lower - margin, 0)
    upper = np.minimum(upper + margin, volume.shape)
    volume = volume[
        lower[0]:upper[0],
        lower[1]:upper[1],
        lower[2]:upper[2],
    ]

    volume = robust_minmax_normalize(volume)
    indices = choose_slice_indices(volume.shape[2])
    selected_slices = []

    for slice_index in indices:
        slice_image = volume[:, :, int(slice_index)]
        resized = cv2.resize(
            slice_image,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_AREA,
        )
        selected_slices.append(resized[..., np.newaxis])

    return np.stack(selected_slices).astype(np.float32)


def cache_signature() -> str:
    return (
        f"{PREPROCESSING_VERSION}_s{N_SLICES}_step{SLICE_STEP}_size{IMG_SIZE}_"
        f"voxel{TARGET_VOXEL_MM:g}_p{LOWER_INTENSITY_PERCENTILE:g}-"
        f"{UPPER_INTENSITY_PERCENTILE:g}"
    )


def get_cache_path(subject_id: int) -> Path:
    return CACHE_DIR / f"{int(subject_id)}_{cache_signature()}.npy"


def load_or_create_cached_slices(
    subject_id: int,
    file_path: str,
) -> np.ndarray:
    cache_path = get_cache_path(subject_id)

    if USE_CACHE and cache_path.exists() and not REBUILD_CACHE:
        return np.load(cache_path).astype(np.float32)

    slices = load_multi_slice_volume(file_path)

    if USE_CACHE:
        np.save(cache_path, slices)

    return slices


# ============================================================
# METADATA PREPROCESSING
# ============================================================

def make_one_hot_encoder() -> OneHotEncoder:
    """
    Supports both newer and older scikit-learn versions.
    """
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def make_metadata_preprocessor(
    train_df: pd.DataFrame,
) -> Tuple[Optional[ColumnTransformer], List[str], List[str]]:
    numeric_columns = [
        column
        for column in NUMERIC_METADATA_COLUMNS
        if column in train_df.columns
    ]

    categorical_columns = [
        column
        for column in CATEGORICAL_METADATA_COLUMNS
        if column in train_df.columns
    ]

    transformers = []

    if numeric_columns:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ])

        transformers.append(
            ("numeric", numeric_pipeline, numeric_columns)
        )

    if categorical_columns:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])

        transformers.append(
            ("categorical", categorical_pipeline, categorical_columns)
        )

    if not transformers:
        return None, numeric_columns, categorical_columns

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    preprocessor.fit(train_df)

    return preprocessor, numeric_columns, categorical_columns


def transform_metadata(
    preprocessor: Optional[ColumnTransformer],
    dataframe: pd.DataFrame,
) -> np.ndarray:
    if preprocessor is None:
        return np.zeros((len(dataframe), 1), dtype=np.float32)

    transformed = preprocessor.transform(dataframe)
    return np.asarray(transformed, dtype=np.float32)


# ============================================================
# SITE ENCODING
# ============================================================

def fit_site_mapping(train_sites: pd.Series) -> Dict[str, int]:
    unique_sites = sorted(train_sites.astype(str).unique())

    return {
        site: index
        for index, site in enumerate(unique_sites)
    }


def encode_sites(
    sites: pd.Series,
    site_mapping: Dict[str, int],
) -> np.ndarray:
    return np.asarray(
        [
            site_mapping.get(str(site), -1)
            for site in sites
        ],
        dtype=np.int32,
    )


# ============================================================
# DATA GENERATOR
# ============================================================

class MapASDDataGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        dataframe: pd.DataFrame,
        metadata_array: np.ndarray,
        site_labels: np.ndarray,
        diagnosis_sample_weights: Optional[np.ndarray] = None,
        batch_size: int = BATCH_SIZE,
        shuffle: bool = True,
        augment: bool = False,
        include_metadata: bool = True,
        include_site_output: bool = False,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.dataframe = dataframe.reset_index(drop=True)
        self.metadata_array = np.asarray(metadata_array, dtype=np.float32)
        self.site_labels = np.asarray(site_labels, dtype=np.int32)

        if diagnosis_sample_weights is None:
            diagnosis_sample_weights = np.ones(
                len(self.dataframe),
                dtype=np.float32,
            )

        self.diagnosis_sample_weights = np.asarray(
            diagnosis_sample_weights,
            dtype=np.float32,
        )

        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.include_metadata = include_metadata
        self.include_site_output = include_site_output

        self.indices = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self) -> int:
        return int(np.ceil(len(self.dataframe) / self.batch_size))

    def __getitem__(self, batch_index: int):
        selected_indices = self.indices[
            batch_index * self.batch_size:
            (batch_index + 1) * self.batch_size
        ]

        batch_df = self.dataframe.iloc[selected_indices]

        images = []
        diagnosis_labels = []
        site_labels = []
        site_sample_weights = []

        for selected_index, (_, row) in zip(
            selected_indices,
            batch_df.iterrows(),
        ):
            image = load_or_create_cached_slices(
                subject_id=int(row["subject_id"]),
                file_path=str(row["file_path"]),
            )

            if self.augment:
                image = self.augment_volume(image)

            images.append(image)
            diagnosis_labels.append(float(row["label"]))

            site_label = int(self.site_labels[selected_index])

            if site_label < 0:
                site_labels.append(0)
                site_sample_weights.append(0.0)
            else:
                site_labels.append(site_label)
                site_sample_weights.append(1.0)

        images = np.asarray(images, dtype=np.float32)
        diagnosis_labels = np.asarray(
            diagnosis_labels,
            dtype=np.float32,
        )

        inputs = {"mri_input": images}

        if self.include_metadata:
            inputs["metadata_input"] = self.metadata_array[selected_indices]

        if not self.include_site_output:
            return (
                inputs,
                diagnosis_labels,
                self.diagnosis_sample_weights[selected_indices],
            )

        outputs = {
            "diagnosis_output": diagnosis_labels,
            "site_output": np.asarray(site_labels, dtype=np.int32),
        }

        sample_weights = {
            "diagnosis_output":
                self.diagnosis_sample_weights[selected_indices],
            "site_output":
                np.asarray(site_sample_weights, dtype=np.float32),
        }

        return inputs, outputs, sample_weights

    def on_epoch_end(self) -> None:
        if self.shuffle:
            np.random.shuffle(self.indices)

    @staticmethod
    def augment_volume(volume: np.ndarray) -> np.ndarray:
        """
        Apply one consistent geometric transform to every slice.
        No left-right or superior-inferior flipping is used.
        """
        volume = volume.copy()

        angle = np.random.uniform(-5.0, 5.0)
        scale = np.random.uniform(0.95, 1.05)
        tx = np.random.uniform(-4.0, 4.0)
        ty = np.random.uniform(-4.0, 4.0)

        centre = (IMG_SIZE / 2.0, IMG_SIZE / 2.0)

        matrix = cv2.getRotationMatrix2D(
            centre,
            angle,
            scale,
        )

        matrix[:, 2] += [tx, ty]

        transformed_slices = []

        for slice_image in volume:
            transformed = cv2.warpAffine(
                slice_image[..., 0],
                matrix,
                (IMG_SIZE, IMG_SIZE),
                flags=cv2.INTER_LINEAR,
                borderMode=cv2.BORDER_REFLECT_101,
            )

            transformed_slices.append(
                transformed[..., np.newaxis]
            )

        transformed_volume = np.stack(transformed_slices)

        intensity_scale = np.random.uniform(0.95, 1.05)
        intensity_shift = np.random.uniform(-0.03, 0.03)

        transformed_volume = (
            transformed_volume * intensity_scale
            + intensity_shift
        )

        return np.clip(
            transformed_volume,
            0.0,
            1.0,
        ).astype(np.float32)


# ============================================================
# CUSTOM LAYERS
# ============================================================

@tf.keras.utils.register_keras_serializable()
class SliceAttention(layers.Layer):

    def __init__(
        self,
        attention_units: int = 64,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.attention_units = attention_units
        self.projection = layers.Dense(
            attention_units,
            activation="tanh",
        )
        self.score = layers.Dense(1)

    def call(self, inputs):
        hidden = self.projection(inputs)
        logits = self.score(hidden)
        weights = tf.nn.softmax(logits, axis=1)
        return tf.reduce_sum(inputs * weights, axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({"attention_units": self.attention_units})
        return config


@tf.keras.utils.register_keras_serializable()
class GradientReversal(layers.Layer):

    def __init__(
        self,
        coefficient: float = 0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.initial_coefficient = float(coefficient)

    def build(self, input_shape):
        self.coefficient = self.add_weight(
            name="grl_coefficient",
            shape=(),
            initializer=tf.keras.initializers.Constant(
                self.initial_coefficient
            ),
            trainable=False,
        )
        super().build(input_shape)

    def call(self, inputs):
        coefficient = tf.cast(self.coefficient, inputs.dtype)

        @tf.custom_gradient
        def reverse_gradient(x):
            def gradient(dy):
                return -coefficient * dy
            return x, gradient

        return reverse_gradient(inputs)

    def get_config(self):
        config = super().get_config()
        config.update({
            "coefficient": self.initial_coefficient,
        })
        return config


class GradientReversalScheduler(tf.keras.callbacks.Callback):

    def __init__(
        self,
        layer_name: str = "gradient_reversal",
        maximum_coefficient: float = GRL_MAX_COEFFICIENT,
        warmup_epochs: int = GRL_WARMUP_EPOCHS,
        total_epochs: int = EPOCHS,
    ):
        super().__init__()
        self.layer_name = layer_name
        self.maximum_coefficient = maximum_coefficient
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs

    def on_epoch_begin(self, epoch, logs=None):
        grl_layer = self.model.get_layer(self.layer_name)

        if epoch < self.warmup_epochs:
            coefficient = 0.0
        else:
            denominator = max(
                self.total_epochs - self.warmup_epochs - 1,
                1,
            )
            progress = (epoch - self.warmup_epochs) / denominator
            progress = float(np.clip(progress, 0.0, 1.0))
            coefficient = self.maximum_coefficient * progress

        grl_layer.coefficient.assign(coefficient)

        print(
            f"\n[GRL] epoch={epoch + 1}, "
            f"coefficient={coefficient:.5f}"
        )


# ============================================================
# MODEL COMPONENTS
# ============================================================

def build_slice_cnn() -> tf.keras.Model:
    image_input = layers.Input(
        shape=(IMG_SIZE, IMG_SIZE, 1),
        name="slice_input",
    )

    x = image_input

    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(
            filters,
            kernel_size=3,
            padding="same",
            use_bias=False,
            kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(pool_size=2)(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
        name="slice_embedding",
    )(x)

    return models.Model(
        image_input,
        x,
        name="shared_slice_cnn",
    )


def build_mapasdnet(
    metadata_dim: int,
    num_sites: int,
    use_cnn: bool = True,
    recurrent_type: str = "bilstm",
    use_attention: bool = True,
    use_metadata: bool = True,
    use_adversarial: bool = True,
) -> tf.keras.Model:

    mri_input = layers.Input(
        shape=(N_SLICES, IMG_SIZE, IMG_SIZE, 1),
        name="mri_input",
    )

    # --------------------------------------------------------
    # SLICE FEATURE EXTRACTION
    # --------------------------------------------------------

    if use_cnn:
        slice_encoder = build_slice_cnn()

        sequence = layers.TimeDistributed(
            slice_encoder,
            name="slice_feature_extraction",
        )(mri_input)

    else:
        # Explicit BiLSTM-only ablation:
        # resize -> flatten -> shared projection -> BiLSTM
        resized = layers.TimeDistributed(
            layers.Resizing(32, 32),
            name="bilstm_only_resize",
        )(mri_input)

        flattened = layers.TimeDistributed(
            layers.Flatten(),
            name="bilstm_only_flatten",
        )(resized)

        sequence = layers.TimeDistributed(
            layers.Dense(
                128,
                activation="relu",
                kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
            ),
            name="bilstm_only_projection",
        )(flattened)

    # --------------------------------------------------------
    # INTER-SLICE MODELLING
    # --------------------------------------------------------
    # recurrent_type controls the recurrent ablation:
    #   "none"   -> no recurrent layer (CNN-only)
    #   "lstm"   -> forward LSTM
    #   "bilstm" -> bidirectional LSTM

    if recurrent_type == "lstm":
        sequence = layers.LSTM(
            96,
            return_sequences=True,
            dropout=LSTM_DROPOUT,
            name="inter_slice_lstm",
        )(sequence)

    elif recurrent_type == "bilstm":
        sequence = layers.Bidirectional(
            layers.LSTM(
                96,
                return_sequences=True,
                dropout=LSTM_DROPOUT,
            ),
            name="inter_slice_bilstm",
        )(sequence)

    elif recurrent_type == "none":
        pass

    else:
        raise ValueError(
            "recurrent_type must be one of: 'none', 'lstm', 'bilstm'. "
            f"Received: {recurrent_type}"
        )

    if recurrent_type in {"lstm", "bilstm"}:
        if use_attention:
            image_features = SliceAttention(
                attention_units=64,
                name="slice_attention",
            )(sequence)
        else:
            image_features = layers.GlobalAveragePooling1D(
                name="temporal_average_pooling"
            )(sequence)
    else:
        image_features = layers.GlobalAveragePooling1D(
            name="slice_average_pooling"
        )(sequence)

    image_features = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
        name="image_projection",
    )(image_features)

    image_features = layers.BatchNormalization(
        name="image_batch_normalization"
    )(image_features)

    image_features = layers.Dropout(
        IMAGE_DROPOUT,
        name="image_dropout",
    )(image_features)

    model_inputs = [mri_input]
    fused_features = image_features

    # --------------------------------------------------------
    # METADATA FUSION
    # --------------------------------------------------------

    if use_metadata:
        metadata_input = layers.Input(
            shape=(metadata_dim,),
            name="metadata_input",
        )

        metadata_features = layers.Dense(
            32,
            activation="relu",
            kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
            name="metadata_dense",
        )(metadata_input)

        metadata_features = layers.BatchNormalization(
            name="metadata_batch_normalization"
        )(metadata_features)

        metadata_features = layers.Dropout(
            METADATA_DROPOUT,
            name="metadata_dropout",
        )(metadata_features)

        fused_features = layers.Concatenate(
            name="image_metadata_fusion"
        )([image_features, metadata_features])

        model_inputs.append(metadata_input)

    # --------------------------------------------------------
    # SHARED DIAGNOSTIC REPRESENTATION
    # --------------------------------------------------------

    shared_features = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
        name="shared_dense",
    )(fused_features)

    shared_features = layers.BatchNormalization(
        name="shared_batch_normalization"
    )(shared_features)

    shared_features = layers.Dropout(
        SHARED_DROPOUT,
        name="shared_dropout",
    )(shared_features)

    diagnosis_output = layers.Dense(
        1,
        activation="sigmoid",
        name="diagnosis_output",
    )(shared_features)

    # BCE is more stable here than combining focal loss with class weights.
    diagnosis_loss = BinaryCrossentropy()

    diagnosis_metrics = [
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
    ]

    # --------------------------------------------------------
    # OPTIONAL SITE-ADVERSARIAL BRANCH
    # --------------------------------------------------------

    if use_adversarial:
        reversed_features = GradientReversal(
            coefficient=0.0,
            name="gradient_reversal",
        )(shared_features)

        site_features = layers.Dense(
            64,
            activation="relu",
            name="site_dense",
        )(reversed_features)

        site_features = layers.Dropout(
            SITE_DROPOUT,
            name="site_dropout",
        )(site_features)

        site_output = layers.Dense(
            max(num_sites, 2),
            activation="softmax",
            name="site_output",
        )(site_features)

        model = models.Model(
            inputs=model_inputs,
            outputs={
                "diagnosis_output": diagnosis_output,
                "site_output": site_output,
            },
            name="MapASDNet",
        )

        model.compile(
            optimizer=Adam(learning_rate=LEARNING_RATE),
            loss={
                "diagnosis_output": diagnosis_loss,
                "site_output":
                    tf.keras.losses.SparseCategoricalCrossentropy(),
            },
            loss_weights={
                "diagnosis_output": 1.0,
                "site_output": SITE_LOSS_WEIGHT,
            },
            metrics={
                "diagnosis_output": diagnosis_metrics,
                "site_output": [
                    tf.keras.metrics.SparseCategoricalAccuracy(
                        name="accuracy"
                    )
                ],
            },
        )

    else:
        model = models.Model(
            inputs=model_inputs,
            outputs=diagnosis_output,
            name="MapASDNet",
        )

        model.compile(
            optimizer=Adam(learning_rate=LEARNING_RATE),
            loss=diagnosis_loss,
            metrics=diagnosis_metrics,
        )

    return model


# ============================================================
# EXPERIMENT CONFIGURATIONS
# ============================================================

ABLATION_CONFIGS = {
    # --------------------------------------------------------
    # 1. CNN-only ablation
    # Multi-slice shared CNN + slice-average pooling.
    # No recurrent layer, attention, metadata or adversarial branch.
    # --------------------------------------------------------
    "cnn_only": {
        "use_cnn": True,
        "recurrent_type": "none",
        "use_attention": False,
        "use_metadata": False,
        "use_adversarial": False,
    },

    # --------------------------------------------------------
    # 2. LSTM-only ablation
    # Slices are resized, flattened and projected before a forward LSTM.
    # No CNN, attention, metadata or adversarial branch.
    # --------------------------------------------------------
    "lstm_only": {
        "use_cnn": False,
        "recurrent_type": "lstm",
        "use_attention": False,
        "use_metadata": False,
        "use_adversarial": False,
    },

    # --------------------------------------------------------
    # 3. BiLSTM-only ablation
    # Slices are resized, flattened and projected before a BiLSTM.
    # No CNN, attention, metadata or adversarial branch.
    # --------------------------------------------------------
    "bilstm_only": {
        "use_cnn": False,
        "recurrent_type": "bilstm",
        "use_attention": False,
        "use_metadata": False,
        "use_adversarial": False,
    },

    # --------------------------------------------------------
    # 4. CNN-LSTM ablation
    # Shared CNN slice features + forward inter-slice LSTM.
    # --------------------------------------------------------
    "cnn_lstm": {
        "use_cnn": True,
        "recurrent_type": "lstm",
        "use_attention": False,
        "use_metadata": False,
        "use_adversarial": False,
    },

    # --------------------------------------------------------
    # 5. CNN-BiLSTM baseline -- RUN THIS FIRST
    # Shared CNN slice features + bidirectional inter-slice modelling.
    # --------------------------------------------------------
    "cnn_bilstm": {
        "use_cnn": True,
        "recurrent_type": "bilstm",
        "use_attention": False,
        "use_metadata": False,
        "use_adversarial": False,
    },

    # --------------------------------------------------------
    # 6. Attention ablation
    # Adds slice attention to the CNN-BiLSTM baseline.
    # --------------------------------------------------------
    "cnn_bilstm_attention": {
        "use_cnn": True,
        "recurrent_type": "bilstm",
        "use_attention": True,
        "use_metadata": False,
        "use_adversarial": False,
    },

    # --------------------------------------------------------
    # 7. Metadata-fusion ablation
    # Adds age, IQ measures and sex to CNN-BiLSTM-attention.
    # --------------------------------------------------------
    "cnn_bilstm_attention_metadata": {
        "use_cnn": True,
        "recurrent_type": "bilstm",
        "use_attention": True,
        "use_metadata": True,
        "use_adversarial": False,
    },

    # --------------------------------------------------------
    # 8. Full adversarial model
    # Adds site-adversarial learning through gradient reversal.
    # --------------------------------------------------------
    "cnn_bilstm_attention_metadata_adversarial": {
        "use_cnn": True,
        "recurrent_type": "bilstm",
        "use_attention": True,
        "use_metadata": True,
        "use_adversarial": True,
    },
}



# ============================================================
# METRICS AND THRESHOLDING
# ============================================================

def select_validation_threshold(
    labels: np.ndarray,
    probabilities: np.ndarray,
    objective: str = THRESHOLD_OBJECTIVE,
) -> Tuple[float, float]:
    thresholds = np.arange(
        THRESHOLD_MIN,
        THRESHOLD_MAX + 1e-8,
        THRESHOLD_STEP,
    )

    best_threshold = 0.50
    best_score = -np.inf

    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(np.int32)

        if objective == "accuracy":
            score = accuracy_score(labels, predictions)
        elif objective == "f1":
            score = f1_score(labels, predictions, zero_division=0)
        elif objective == "balanced_accuracy":
            score = balanced_accuracy_score(labels, predictions)
        else:
            raise ValueError(f"Unknown threshold objective: {objective}")

        better = score > best_score + 1e-12
        stable_tie = (
            abs(score - best_score) <= 1e-12
            and abs(threshold - 0.50) < abs(best_threshold - 0.50)
        )
        if better or stable_tie:
            best_score = float(score)
            best_threshold = float(threshold)

    return best_threshold, best_score


def calculate_metrics(
    labels: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    predictions = (probabilities >= threshold).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    try:
        auc = roc_auc_score(labels, probabilities)
    except ValueError:
        auc = np.nan

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(
            balanced_accuracy_score(labels, predictions)
        ),
        "precision": float(
            precision_score(labels, predictions, zero_division=0)
        ),
        "recall": float(
            recall_score(labels, predictions, zero_division=0)
        ),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(
            f1_score(labels, predictions, zero_division=0)
        ),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def get_diagnosis_probabilities(
    model: tf.keras.Model,
    generator: MapASDDataGenerator,
    use_adversarial: bool,
) -> np.ndarray:
    predictions = model.predict(generator, verbose=0)

    if use_adversarial:
        if isinstance(predictions, dict):
            predictions = predictions["diagnosis_output"]
        elif isinstance(predictions, list):
            predictions = predictions[0]

    return np.asarray(predictions).reshape(-1)


def compute_diagnosis_sample_weights(
    labels: np.ndarray,
) -> Tuple[Dict[int, float], np.ndarray]:
    classes = np.unique(labels)

    values = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=labels,
    )

    class_weight_map = {
        int(class_label): float(weight)
        for class_label, weight in zip(classes, values)
    }

    sample_weights = np.asarray(
        [class_weight_map[int(label)] for label in labels],
        dtype=np.float32,
    )

    return class_weight_map, sample_weights


# ============================================================
# CALLBACKS
# ============================================================

def make_callbacks(
    checkpoint_path: Path,
    use_adversarial: bool,
) -> List[tf.keras.callbacks.Callback]:
    callbacks: List[tf.keras.callbacks.Callback] = []

    # Keras reports different metric names for one-output and multi-output
    # models. The adversarial model uses val_diagnosis_output_auc, whereas
    # non-adversarial ablations use val_auc.
    diagnosis_auc_monitor = (
        "val_diagnosis_output_auc"
        if use_adversarial
        else "val_auc"
    )

    if use_adversarial:
        callbacks.append(
            GradientReversalScheduler(
                layer_name="gradient_reversal",
                maximum_coefficient=GRL_MAX_COEFFICIENT,
                warmup_epochs=GRL_WARMUP_EPOCHS,
                total_epochs=EPOCHS,
            )
        )

    callbacks.extend([
        EarlyStopping(
            monitor=diagnosis_auc_monitor,
            mode="max",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),

        ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor=diagnosis_auc_monitor,
            mode="max",
            save_best_only=True,
            verbose=1,
        ),

        ReduceLROnPlateau(
            monitor=diagnosis_auc_monitor,
            mode="max",
            factor=LR_REDUCTION_FACTOR,
            patience=LR_PATIENCE,
            min_lr=MIN_LEARNING_RATE,
            verbose=1,
        ),
    ])

    return callbacks


# ============================================================
# TRAIN ONE SPLIT
# ============================================================

def save_split_membership(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    split_dir: Path,
) -> None:
    train_df.to_csv(split_dir / "train_subjects.csv", index=False)
    valid_df.to_csv(split_dir / "validation_subjects.csv", index=False)
    test_df.to_csv(split_dir / "test_subjects.csv", index=False)


def save_config(
    experiment_name: str,
    config: Dict,
    split_dir: Path,
    run_seed: int = SEED,
) -> None:
    serializable = {
        "experiment_name": experiment_name,
        "seed": run_seed,
        "image_size": IMG_SIZE,
        "n_slices": N_SLICES,
        "slice_step": SLICE_STEP,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "diagnosis_loss": "binary_crossentropy",
        "site_loss_weight": SITE_LOSS_WEIGHT,
        "grl_max_coefficient": GRL_MAX_COEFFICIENT,
        "grl_warmup_epochs": GRL_WARMUP_EPOCHS,
        **config,
    }

    with open(
        split_dir / "experiment_config.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(serializable, file, indent=2)


def train_one_split(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    experiment_name: str,
    config: Dict,
    split_name: str,
    run_seed: int = SEED,
) -> Dict[str, float]:

    tf.keras.backend.clear_session()
    gc.collect()
    set_global_seed(run_seed)

    split_dir = OUTPUT_DIR / experiment_name / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    save_split_membership(
        train_df,
        valid_df,
        test_df,
        split_dir,
    )

    save_config(
        experiment_name,
        config,
        split_dir,
        run_seed=run_seed,
    )

    # --------------------------------------------------------
    # METADATA: fit on training data only
    # --------------------------------------------------------

    metadata_preprocessor, numeric_columns, categorical_columns = (
        make_metadata_preprocessor(train_df)
    )

    train_metadata = transform_metadata(
        metadata_preprocessor,
        train_df,
    )
    valid_metadata = transform_metadata(
        metadata_preprocessor,
        valid_df,
    )
    test_metadata = transform_metadata(
        metadata_preprocessor,
        test_df,
    )

    metadata_dim = int(train_metadata.shape[1])

    # --------------------------------------------------------
    # SITE ENCODING: fit on training sites only
    # --------------------------------------------------------

    site_mapping = fit_site_mapping(train_df["site"])

    train_sites = encode_sites(
        train_df["site"],
        site_mapping,
    )
    valid_sites = encode_sites(
        valid_df["site"],
        site_mapping,
    )
    test_sites = encode_sites(
        test_df["site"],
        site_mapping,
    )

    num_sites = len(site_mapping)

    with open(
        split_dir / "site_mapping.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(site_mapping, file, indent=2)

    # --------------------------------------------------------
    # DIAGNOSIS CLASS WEIGHTING
    # --------------------------------------------------------

    if USE_CLASS_WEIGHTS:
        class_weight_map, train_diagnosis_weights = (
            compute_diagnosis_sample_weights(train_df["label"].to_numpy())
        )
    else:
        class_weight_map = {0: 1.0, 1: 1.0}
        train_diagnosis_weights = np.ones(len(train_df), dtype=np.float32)

    valid_diagnosis_weights = np.ones(
        len(valid_df),
        dtype=np.float32,
    )
    test_diagnosis_weights = np.ones(
        len(test_df),
        dtype=np.float32,
    )

    print("\n" + "=" * 80)
    print(f"Experiment : {experiment_name}")
    print(f"Split      : {split_name}")
    print(f"Train      : {len(train_df)}")
    print(f"Validation : {len(valid_df)}")
    print(f"Test       : {len(test_df)}")
    print(f"Metadata   : {numeric_columns + categorical_columns}")
    print(f"Metadata dimension: {metadata_dim}")
    print(f"Training sites    : {num_sites}")
    print(f"Class weights     : {class_weight_map}")
    print("=" * 80)

    train_generator = MapASDDataGenerator(
        dataframe=train_df,
        metadata_array=train_metadata,
        site_labels=train_sites,
        diagnosis_sample_weights=train_diagnosis_weights,
        batch_size=BATCH_SIZE,
        shuffle=True,
        augment=True,
        include_metadata=config["use_metadata"],
        include_site_output=config["use_adversarial"],
    )

    valid_generator = MapASDDataGenerator(
        dataframe=valid_df,
        metadata_array=valid_metadata,
        site_labels=valid_sites,
        diagnosis_sample_weights=valid_diagnosis_weights,
        batch_size=BATCH_SIZE,
        shuffle=False,
        augment=False,
        include_metadata=config["use_metadata"],
        include_site_output=config["use_adversarial"],
    )

    test_generator = MapASDDataGenerator(
        dataframe=test_df,
        metadata_array=test_metadata,
        site_labels=test_sites,
        diagnosis_sample_weights=test_diagnosis_weights,
        batch_size=BATCH_SIZE,
        shuffle=False,
        augment=False,
        include_metadata=config["use_metadata"],
        include_site_output=config["use_adversarial"],
    )

    model = build_mapasdnet(
        metadata_dim=metadata_dim,
        num_sites=max(num_sites, 2),
        **config,
    )

    model.summary()

    checkpoint_path = split_dir / "best_model.keras"

    callbacks = make_callbacks(
        checkpoint_path=checkpoint_path,
        use_adversarial=config["use_adversarial"],
    )

    history = model.fit(
        train_generator,
        validation_data=valid_generator,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(
        split_dir / "training_history.csv",
        index=False,
    )

    valid_probabilities = get_diagnosis_probabilities(
        model,
        valid_generator,
        config["use_adversarial"],
    )

    test_probabilities = get_diagnosis_probabilities(
        model,
        test_generator,
        config["use_adversarial"],
    )

    valid_labels = valid_df["label"].to_numpy()
    test_labels = test_df["label"].to_numpy()

    threshold, validation_threshold_score = (
        select_validation_threshold(
            labels=valid_labels,
            probabilities=valid_probabilities,
            objective=THRESHOLD_OBJECTIVE,
        )
    )

    metrics = calculate_metrics(
        labels=test_labels,
        probabilities=test_probabilities,
        threshold=threshold,
    )

    metrics.update({
        "experiment": experiment_name,
        "split": split_name,
        "run_seed": int(run_seed),
        "n_train": len(train_df),
        "n_valid": len(valid_df),
        "n_test": len(test_df),
        "validation_threshold_score":
            float(validation_threshold_score),
        "metadata_dim": metadata_dim,
        "n_training_sites": num_sites,
    })

    validation_predictions = valid_df[
        ["subject_id", "site", "label"]
    ].copy()

    validation_predictions["probability"] = valid_probabilities
    validation_predictions["prediction"] = (
        valid_probabilities >= threshold
    ).astype(np.int32)

    validation_predictions.to_csv(
        split_dir / "validation_predictions.csv",
        index=False,
    )

    test_predictions = test_df[
        ["subject_id", "site", "label"]
    ].copy()

    test_predictions["probability"] = test_probabilities
    test_predictions["prediction"] = (
        test_probabilities >= threshold
    ).astype(np.int32)

    test_predictions.to_csv(
        split_dir / "test_predictions.csv",
        index=False,
    )

    pd.DataFrame([metrics]).to_csv(
        split_dir / "metrics.csv",
        index=False,
    )

    model.save(split_dir / "final_model.keras")

    print("\nFinal test metrics:")
    for key, value in metrics.items():
        print(f"{key:28s}: {value}")

    return metrics


# ============================================================
# FIXED 70/20/10 SPLIT
# ============================================================

def create_fixed_split(
    records,
    split_path=SPLITS_DIR / "fixed_70_20_10.csv",
):
    if split_path.exists():
        saved_split = pd.read_csv(split_path)

        merged = records.merge(
            saved_split[["subject_id", "partition"]],
            on="subject_id",
            how="inner",
            validate="one_to_one",
        )

        train_df = merged[
            merged["partition"] == "train"
        ].drop(columns="partition")

        valid_df = merged[
            merged["partition"] == "validation"
        ].drop(columns="partition")

        test_df = merged[
            merged["partition"] == "test"
        ].drop(columns="partition")

        print(f"Loaded fixed split from: {split_path}")

    else:
        train_df, temporary_df = train_test_split(
            records,
            test_size=0.30,
            stratify=records["label"],
            random_state=SEED,
        )

        valid_df, test_df = train_test_split(
            temporary_df,
            test_size=1.0 / 3.0,
            stratify=temporary_df["label"],
            random_state=SEED,
        )

        split_rows = []

        for partition, dataframe in [
            ("train", train_df),
            ("validation", valid_df),
            ("test", test_df),
        ]:
            part = dataframe[
                ["subject_id"]
            ].copy()

            part["partition"] = partition
            split_rows.append(part)

        pd.concat(
            split_rows,
            ignore_index=True,
        ).to_csv(
            split_path,
            index=False,
        )

        print(f"Created fixed split: {split_path}")

    return (
        train_df.reset_index(drop=True),
        valid_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


# ============================================================
# STRATIFIED FIVE-FOLD CROSS-VALIDATION
# ============================================================

def run_cross_validation(
    records: pd.DataFrame,
    experiments: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    if experiments is None:
        experiments = CV_EXPERIMENTS

    splitter = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED,
    )

    all_results: List[Dict] = []

    for fold, (development_indices, test_indices) in enumerate(
        splitter.split(records, records["label"]),
        start=1,
    ):
        development_df = records.iloc[
            development_indices
        ].reset_index(drop=True)

        test_df = records.iloc[
            test_indices
        ].reset_index(drop=True)

        train_df, valid_df = train_test_split(
            development_df,
            test_size=0.15,
            stratify=development_df["label"],
            random_state=SEED + fold,
        )

        train_df = train_df.reset_index(drop=True)
        valid_df = valid_df.reset_index(drop=True)

        for experiment_name in experiments:
            if experiment_name not in ABLATION_CONFIGS:
                raise KeyError(
                    f"Unknown experiment: {experiment_name}"
                )

            metrics = train_one_split(
                train_df=train_df,
                valid_df=valid_df,
                test_df=test_df,
                experiment_name=experiment_name,
                config=ABLATION_CONFIGS[experiment_name],
                split_name=f"fold_{fold}",
            )

            metrics["fold"] = fold
            all_results.append(metrics)

            pd.DataFrame(all_results).to_csv(
                CROSS_VALIDATION_DIR /
                "cross_validation_live_results.csv",
                index=False,
            )

    results_df = pd.DataFrame(all_results)

    results_df.to_csv(
        CROSS_VALIDATION_DIR /
        "cross_validation_all_results.csv",
        index=False,
    )

    metric_columns = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "sensitivity",
        "specificity",
        "f1",
        "auc",
    ]

    summary_df = (
        results_df
        .groupby("experiment")[metric_columns]
        .agg(["mean", "std"])
    )

    summary_df.to_csv(
        CROSS_VALIDATION_DIR /
        "cross_validation_mean_std.csv"
    )

    print("\nCross-validation mean ± standard deviation:")
    print(summary_df)

    return results_df, summary_df


# ============================================================
# PAIRED SIGNIFICANCE TESTING
# ============================================================

def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)

    running_maximum = 0.0
    number_of_tests = len(p_values)

    for rank, index in enumerate(order):
        corrected = (number_of_tests - rank) * p_values[index]
        running_maximum = max(running_maximum, corrected)
        adjusted[index] = min(running_maximum, 1.0)

    return adjusted


def run_paired_significance_tests(
    cv_results: pd.DataFrame,
    reference_experiment: str = "cnn_bilstm_attention_metadata_adversarial",
) -> pd.DataFrame:

    metric_columns = [
        "accuracy",
        "balanced_accuracy",
        "f1",
        "auc",
    ]

    reference = (
        cv_results[
            cv_results["experiment"] == reference_experiment
        ]
        .sort_values("fold")
    )

    rows: List[Dict] = []

    for experiment in sorted(
        cv_results["experiment"].unique()
    ):
        if experiment == reference_experiment:
            continue

        comparison = (
            cv_results[
                cv_results["experiment"] == experiment
            ]
            .sort_values("fold")
        )

        for metric in metric_columns:
            paired = reference[
                ["fold", metric]
            ].merge(
                comparison[["fold", metric]],
                on="fold",
                suffixes=("_reference", "_comparison"),
            ).dropna()

            if len(paired) < 2:
                continue

            reference_values = paired[
                f"{metric}_reference"
            ].to_numpy()

            comparison_values = paired[
                f"{metric}_comparison"
            ].to_numpy()

            differences = reference_values - comparison_values

            if np.allclose(differences, 0):
                statistic = 0.0
                p_value = 1.0
            else:
                statistic, p_value = wilcoxon(
                    reference_values,
                    comparison_values,
                    alternative="two-sided",
                    zero_method="wilcox",
                )

            rows.append({
                "reference_experiment": reference_experiment,
                "comparison_experiment": experiment,
                "metric": metric,
                "n_pairs": len(paired),
                "reference_mean": float(
                    reference_values.mean()
                ),
                "comparison_mean": float(
                    comparison_values.mean()
                ),
                "mean_difference": float(
                    differences.mean()
                ),
                "median_difference": float(
                    np.median(differences)
                ),
                "wilcoxon_statistic": float(statistic),
                "p_value": float(p_value),
            })

    significance_df = pd.DataFrame(rows)

    if not significance_df.empty:
        significance_df["holm_adjusted_p"] = holm_adjust(
            significance_df["p_value"].to_numpy()
        )

    significance_df.to_csv(
        ABLATION_DIR /
        "paired_significance_tests.csv",
        index=False,
    )

    return significance_df


# ============================================================
# LEAVE-ONE-SITE-OUT
# ============================================================

def run_loso(
    records: pd.DataFrame,
    experiment_name: str = LOSO_EXPERIMENT,
) -> pd.DataFrame:

    if experiment_name not in ABLATION_CONFIGS:
        raise KeyError(f"Unknown experiment: {experiment_name}")

    results: List[Dict] = []
    sites = sorted(records["site"].astype(str).unique())

    for site_index, held_out_site in enumerate(sites, start=1):
        test_df = records[
            records["site"].astype(str) == held_out_site
        ].reset_index(drop=True)

        development_df = records[
            records["site"].astype(str) != held_out_site
        ].reset_index(drop=True)

        if test_df["label"].nunique() < 2:
            print(
                f"\nSkipping LOSO site {held_out_site}: "
                "the held-out site contains only one diagnosis class."
            )
            continue

        if development_df["label"].nunique() < 2:
            continue

        train_df, valid_df = train_test_split(
            development_df,
            test_size=0.15,
            stratify=development_df["label"],
            random_state=SEED + site_index,
        )

        train_df = train_df.reset_index(drop=True)
        valid_df = valid_df.reset_index(drop=True)

        safe_site_name = re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            str(held_out_site),
        ).strip("_")

        metrics = train_one_split(
            train_df=train_df,
            valid_df=valid_df,
            test_df=test_df,
            experiment_name=experiment_name,
            config=ABLATION_CONFIGS[experiment_name],
            split_name=f"loso_{safe_site_name}",
        )

        metrics["held_out_site"] = held_out_site
        results.append(metrics)

        pd.DataFrame(results).to_csv(
            LOSO_DIR / "loso_live_results.csv",
            index=False,
        )

    loso_df = pd.DataFrame(results)

    loso_df.to_csv(
        LOSO_DIR / "loso_all_results.csv",
        index=False,
    )

    metric_columns = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "sensitivity",
        "specificity",
        "f1",
        "auc",
    ]

    if not loso_df.empty:
        loso_summary = (
            loso_df[metric_columns]
            .agg(["mean", "std", "min", "max"])
            .transpose()
        )

        loso_summary.to_csv(
            LOSO_DIR / "loso_summary.csv"
        )

        print("\nLOSO summary:")
        print(loso_summary)

    return loso_df



Detected TensorFlow GPUs:
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  - PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
  - PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')
  - PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')


## Dataset preparation and fixed 70/15/15 split

In [ ]:
# ============================================================
# CREATE/LOAD ONE PERSISTENT 70/15/15 SPLIT
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

SPLIT_NAME = "fixed_70_15_15"
FIXED_SPLIT_FILE = SPLITS_DIR / "fixed_70_15_15.csv"


def create_or_load_fixed_split_70_15_15(records: pd.DataFrame):
    """Create exactly one subject-level 70/15/15 split and reuse it for every ablation."""
    current_ids = set(records["subject_id"].astype(int))

    if FIXED_SPLIT_FILE.exists():
        saved = pd.read_csv(FIXED_SPLIT_FILE)
        required = {"subject_id", "partition"}
        if not required.issubset(saved.columns):
            raise ValueError(f"Invalid split file: {FIXED_SPLIT_FILE}")

        saved_ids = set(saved["subject_id"].astype(int))
        if saved_ids != current_ids:
            raise ValueError(
                "The existing 70/15/15 split does not match the current matched-subject table. "
                "Delete the split file only if the dataset selection intentionally changed."
            )

        merged = records.merge(
            saved[["subject_id", "partition"]],
            on="subject_id",
            how="inner",
            validate="one_to_one",
        )
        train_df = merged[merged["partition"] == "train"].drop(columns="partition")
        valid_df = merged[merged["partition"] == "validation"].drop(columns="partition")
        test_df = merged[merged["partition"] == "test"].drop(columns="partition")
        print(f"Loaded persistent split: {FIXED_SPLIT_FILE}")
    else:
        train_df, temp_df = train_test_split(
            records,
            test_size=0.30,
            stratify=records["label"],
            random_state=SEED,
        )
        valid_df, test_df = train_test_split(
            temp_df,
            test_size=0.50,
            stratify=temp_df["label"],
            random_state=SEED,
        )

        parts = []
        for name, frame in [
            ("train", train_df),
            ("validation", valid_df),
            ("test", test_df),
        ]:
            part = frame[["subject_id"]].copy()
            part["partition"] = name
            parts.append(part)

        pd.concat(parts, ignore_index=True).to_csv(FIXED_SPLIT_FILE, index=False)
        print(f"Created persistent split: {FIXED_SPLIT_FILE}")

    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    # Leakage checks
    train_ids = set(train_df["subject_id"])
    valid_ids = set(valid_df["subject_id"])
    test_ids = set(test_df["subject_id"])
    assert train_ids.isdisjoint(valid_ids)
    assert train_ids.isdisjoint(test_ids)
    assert valid_ids.isdisjoint(test_ids)
    assert len(train_df) + len(valid_df) + len(test_df) == len(records)

    total = len(records)
    print("\nPersistent data split")
    print(f"Train      : {len(train_df):4d} ({100*len(train_df)/total:.2f}%)")
    print(f"Validation : {len(valid_df):4d} ({100*len(valid_df)/total:.2f}%)")
    print(f"Test       : {len(test_df):4d} ({100*len(test_df)/total:.2f}%)")
    print(f"N_SLICES   : {N_SLICES}")

    return train_df, valid_df, test_df


existing_folders = validate_paths()
records = create_subject_table(CSV_PATH, existing_folders)
train_df, valid_df, test_df = create_or_load_fixed_split_70_15_15(records)


## Result-export helpers

In [ ]:
# ============================================================
# PER-EXPERIMENT XLSX AND FIGURE EXPORT
# ============================================================


def _history_key(history_df, candidates):
    for key in candidates:
        if key in history_df.columns:
            return key
    return None


def _threshold_table(labels, probabilities):
    rows = []
    for threshold in np.arange(
        THRESHOLD_MIN,
        THRESHOLD_MAX + 1e-8,
        THRESHOLD_STEP,
    ):
        preds = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        rows.append({
            "threshold": float(threshold),
            "accuracy": accuracy_score(labels, preds),
            "balanced_accuracy": balanced_accuracy_score(labels, preds),
            "precision": precision_score(labels, preds, zero_division=0),
            "sensitivity": sensitivity,
            "specificity": specificity,
            "f1": f1_score(labels, preds, zero_division=0),
        })
    return pd.DataFrame(rows)


def export_experiment_artifacts(experiment_name: str, split_name: str = SPLIT_NAME):
    split_dir = OUTPUT_DIR / experiment_name / split_name
    history_path = split_dir / "training_history.csv"
    metrics_path = split_dir / "metrics.csv"
    validation_path = split_dir / "validation_predictions.csv"
    test_path = split_dir / "test_predictions.csv"

    for required in [history_path, metrics_path, validation_path, test_path]:
        if not required.exists():
            raise FileNotFoundError(f"Missing experiment output: {required}")

    history_df = pd.read_csv(history_path)
    metrics_df = pd.read_csv(metrics_path)
    validation_df = pd.read_csv(validation_path)
    test_predictions_df = pd.read_csv(test_path)

    accuracy_key = _history_key(
        history_df,
        ["accuracy", "diagnosis_output_accuracy"],
    )
    val_accuracy_key = _history_key(
        history_df,
        ["val_accuracy", "val_diagnosis_output_accuracy"],
    )
    loss_key = _history_key(
        history_df,
        ["diagnosis_output_loss", "loss"],
    )
    val_loss_key = _history_key(
        history_df,
        ["val_diagnosis_output_loss", "val_loss"],
    )

    epochs = np.arange(1, len(history_df) + 1)

    # Accuracy curve
    if accuracy_key and val_accuracy_key:
        plt.figure(figsize=(8, 5))
        plt.plot(epochs, history_df[accuracy_key], label="Training accuracy")
        plt.plot(epochs, history_df[val_accuracy_key], label="Validation accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title(f"{experiment_name}: training vs validation accuracy")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(split_dir / "training_validation_accuracy.png", dpi=300)
        plt.show()
        plt.close()

    # Loss curve
    if loss_key and val_loss_key:
        plt.figure(figsize=(8, 5))
        plt.plot(epochs, history_df[loss_key], label="Training loss")
        plt.plot(epochs, history_df[val_loss_key], label="Validation loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"{experiment_name}: training vs validation loss")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(split_dir / "training_validation_loss.png", dpi=300)
        plt.show()
        plt.close()

    y_true = test_predictions_df["label"].to_numpy(dtype=int)
    y_prob = test_predictions_df["probability"].to_numpy(dtype=float)
    y_pred = test_predictions_df["prediction"].to_numpy(dtype=int)

    fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
    test_auc = roc_auc_score(y_true, y_prob)
    roc_df = pd.DataFrame({
        "false_positive_rate": fpr,
        "true_positive_rate": tpr,
        "threshold": roc_thresholds,
    })

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"AUC = {test_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"{experiment_name}: test ROC curve")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(split_dir / "test_roc_curve.png", dpi=300)
    plt.show()
    plt.close()

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=["Actual TD", "Actual ASD"],
        columns=["Predicted TD", "Predicted ASD"],
    )

    plt.figure(figsize=(5.5, 4.5))
    plt.imshow(cm)
    plt.xticks([0, 1], ["TD", "ASD"])
    plt.yticks([0, 1], ["TD", "ASD"])
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title(f"{experiment_name}: test confusion matrix")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(split_dir / "test_confusion_matrix.png", dpi=300)
    plt.show()
    plt.close()

    validation_threshold_df = _threshold_table(
        validation_df["label"].to_numpy(dtype=int),
        validation_df["probability"].to_numpy(dtype=float),
    )

    split_summary_df = pd.DataFrame([
        {"partition": "train", "subjects": len(train_df), "ASD": int(train_df["label"].sum()), "TD": int((train_df["label"] == 0).sum())},
        {"partition": "validation", "subjects": len(valid_df), "ASD": int(valid_df["label"].sum()), "TD": int((valid_df["label"] == 0).sum())},
        {"partition": "test", "subjects": len(test_df), "ASD": int(test_df["label"].sum()), "TD": int((test_df["label"] == 0).sum())},
    ])

    config_df = pd.DataFrame([
        {"parameter": "experiment", "value": experiment_name},
        {"parameter": "seed", "value": SEED},
        {"parameter": "image_size", "value": IMG_SIZE},
        {"parameter": "n_slices", "value": N_SLICES},
        {"parameter": "slice_step", "value": SLICE_STEP},
        {"parameter": "batch_size", "value": BATCH_SIZE},
        {"parameter": "epochs", "value": EPOCHS},
        {"parameter": "learning_rate", "value": LEARNING_RATE},
        {"parameter": "split", "value": "70/15/15"},
    ] + [
        {"parameter": key, "value": value}
        for key, value in ABLATION_CONFIGS[experiment_name].items()
    ])

    workbook_path = split_dir / f"{experiment_name}_results.xlsx"
    with pd.ExcelWriter(workbook_path) as writer:
        metrics_df.to_excel(writer, sheet_name="Summary_Metrics", index=False)
        history_df.to_excel(writer, sheet_name="Training_History", index=False)
        validation_df.to_excel(writer, sheet_name="Validation_Predictions", index=False)
        test_predictions_df.to_excel(writer, sheet_name="Test_Predictions", index=False)
        cm_df.to_excel(writer, sheet_name="Confusion_Matrix")
        roc_df.to_excel(writer, sheet_name="ROC_Data", index=False)
        validation_threshold_df.to_excel(writer, sheet_name="Threshold_Analysis", index=False)
        split_summary_df.to_excel(writer, sheet_name="Split_Summary", index=False)
        config_df.to_excel(writer, sheet_name="Configuration", index=False)

    print(f"\nSaved XLSX: {workbook_path}")
    print(f"Saved plots in: {split_dir}")
    return metrics_df.iloc[0].to_dict()


def run_fixed_experiment(experiment_name: str):
    """Train one model on the persistent 70/15/15 split and export all results."""
    if experiment_name not in ABLATION_CONFIGS:
        raise KeyError(f"Unknown experiment: {experiment_name}")

    print("\n" + "=" * 80)
    print("RUNNING EXPERIMENT:", experiment_name)
    print("N_SLICES:", N_SLICES)
    print("SPLIT: 70% train / 15% validation / 15% test")
    print(json.dumps(ABLATION_CONFIGS[experiment_name], indent=2))
    print("=" * 80)

    metrics = train_one_split(
        train_df=train_df,
        valid_df=valid_df,
        test_df=test_df,
        experiment_name=experiment_name,
        config=ABLATION_CONFIGS[experiment_name],
        split_name=SPLIT_NAME,
    )

    export_experiment_artifacts(experiment_name, SPLIT_NAME)
    return metrics



def run_target_seed_ensemble(
    experiment_name: str = "cnn_bilstm_attention_metadata",
    seeds=ENSEMBLE_SEEDS,
):
    """Train predetermined seeds and average probabilities before thresholding.

    The threshold is selected once from averaged validation probabilities. Test
    probabilities are never used for model, seed, or threshold selection.
    """
    if experiment_name != "cnn_bilstm_attention_metadata":
        raise ValueError("This helper is reserved for the target metadata model.")

    validation_probabilities = []
    test_probabilities = []
    validation_reference = None
    test_reference = None

    for run_seed in seeds:
        member_split = (
            SPLIT_NAME if int(run_seed) == int(SEED)
            else f"{SPLIT_NAME}_seed_{int(run_seed)}"
        )
        print(f"\nTraining ensemble member seed={run_seed}")
        train_one_split(
            train_df=train_df,
            valid_df=valid_df,
            test_df=test_df,
            experiment_name=experiment_name,
            config=ABLATION_CONFIGS[experiment_name],
            split_name=member_split,
            run_seed=int(run_seed),
        )

        member_dir = OUTPUT_DIR / experiment_name / member_split
        validation_member = pd.read_csv(member_dir / "validation_predictions.csv")
        test_member = pd.read_csv(member_dir / "test_predictions.csv")

        if validation_reference is None:
            validation_reference = validation_member[["subject_id", "site", "label"]].copy()
            test_reference = test_member[["subject_id", "site", "label"]].copy()
        else:
            if not np.array_equal(
                validation_reference["subject_id"].to_numpy(),
                validation_member["subject_id"].to_numpy(),
            ):
                raise RuntimeError("Validation subject order differs across seeds.")
            if not np.array_equal(
                test_reference["subject_id"].to_numpy(),
                test_member["subject_id"].to_numpy(),
            ):
                raise RuntimeError("Test subject order differs across seeds.")

        validation_probabilities.append(validation_member["probability"].to_numpy())
        test_probabilities.append(test_member["probability"].to_numpy())

    mean_validation_probability = np.mean(validation_probabilities, axis=0)
    mean_test_probability = np.mean(test_probabilities, axis=0)
    threshold, validation_score = select_validation_threshold(
        validation_reference["label"].to_numpy(),
        mean_validation_probability,
        objective=THRESHOLD_OBJECTIVE,
    )
    ensemble_metrics = calculate_metrics(
        test_reference["label"].to_numpy(),
        mean_test_probability,
        threshold,
    )
    ensemble_metrics.update({
        "experiment": f"{experiment_name}_seed_ensemble",
        "split": SPLIT_NAME,
        "seeds": ",".join(str(int(seed)) for seed in seeds),
        "n_models": len(seeds),
        "n_train": len(train_df),
        "n_valid": len(valid_df),
        "n_test": len(test_df),
        "validation_threshold_score": float(validation_score),
    })

    ensemble_dir = OUTPUT_DIR / experiment_name / f"{SPLIT_NAME}_ensemble"
    ensemble_dir.mkdir(parents=True, exist_ok=True)
    validation_reference["probability"] = mean_validation_probability
    validation_reference["prediction"] = (
        mean_validation_probability >= threshold
    ).astype(np.int32)
    test_reference["probability"] = mean_test_probability
    test_reference["prediction"] = (
        mean_test_probability >= threshold
    ).astype(np.int32)
    validation_reference.to_csv(ensemble_dir / "validation_predictions.csv", index=False)
    test_reference.to_csv(ensemble_dir / "test_predictions.csv", index=False)
    pd.DataFrame([ensemble_metrics]).to_csv(ensemble_dir / "metrics.csv", index=False)
    pd.DataFrame({
        f"seed_{int(seed)}": probabilities
        for seed, probabilities in zip(seeds, test_probabilities)
    }).assign(
        ensemble_probability=mean_test_probability,
        label=test_reference["label"].to_numpy(),
    ).to_csv(ensemble_dir / "member_test_probabilities.csv", index=False)

    print("\nEnsemble test metrics:")
    for key, value in ensemble_metrics.items():
        print(f"{key:28s}: {value}")
    return ensemble_metrics


## Experiment 1: CNN-only

In [ ]:
# CNN-only
cnn_only_metrics = run_fixed_experiment("cnn_only")
cnn_only_metrics

## Experiment 2: LSTM-only

In [ ]:
# LSTM-only
lstm_only_metrics = run_fixed_experiment("lstm_only")
lstm_only_metrics

## Experiment 3: BiLSTM-only

In [ ]:
# BiLSTM-only
bilstm_only_metrics = run_fixed_experiment("bilstm_only")
bilstm_only_metrics

## Experiment 4: CNN–LSTM

Run this cell only when you are ready to train this configuration.

In [ ]:
# CNN–LSTM
cnn_lstm_metrics = run_fixed_experiment("cnn_lstm")
cnn_lstm_metrics

## Experiment 5: CNN–BiLSTM

In [ ]:
# CNN–BiLSTM
cnn_bilstm_metrics = run_fixed_experiment("cnn_bilstm")
cnn_bilstm_metrics

## Experiment 6: CNN–BiLSTM + Attention

In [ ]:
# CNN–BiLSTM + Attention
cnn_bilstm_attention_metrics = run_fixed_experiment("cnn_bilstm_attention")
cnn_bilstm_attention_metrics

## Experiment 7: CNN–BiLSTM + attention + metadata ensemble

In [ ]:
#CNN–BiLSTM + attention + metadata
cnn_bilstm_attention_metadata_ensemble_metrics = run_target_seed_ensemble("cnn_bilstm_attention_metadata")
cnn_bilstm_attention_metadata_ensemble_metrics


## Experiment 8:  CNN–BiLSTM + attention + metadata ensemble + Site-Adversarial Learning

In [ ]:
#  CNN–BiLSTM + attention + metadata ensemble + Site-Adversarial Learning
adversarial_full_metrics = run_fixed_experiment("cnn_bilstm_attention_metadata_adversarial")
adversarial_full_metrics

## Consolidated ablation summary

In [ ]:
# ============================================================
# COMBINE ALL COMPLETED FIXED-SPLIT RESULTS INTO ONE XLSX
# ============================================================

completed_rows = []
for experiment_name in ABLATION_CONFIGS:
    metrics_path = OUTPUT_DIR / experiment_name / SPLIT_NAME / "metrics.csv"
    if metrics_path.exists():
        row = pd.read_csv(metrics_path).iloc[0].to_dict()
        completed_rows.append(row)

ensemble_metrics_path = (
    OUTPUT_DIR / "cnn_bilstm_attention_metadata" /
    f"{SPLIT_NAME}_ensemble" / "metrics.csv"
)
if ensemble_metrics_path.exists():
    completed_rows.append(pd.read_csv(ensemble_metrics_path).iloc[0].to_dict())

if completed_rows:
    comparison_df = pd.DataFrame(completed_rows)
    preferred_columns = [
        "experiment", "accuracy", "balanced_accuracy", "precision",
        "recall", "sensitivity", "specificity", "f1", "auc",
        "threshold", "tn", "fp", "fn", "tp",
        "n_train", "n_valid", "n_test",
    ]
    comparison_df = comparison_df[
        [column for column in preferred_columns if column in comparison_df.columns]
    ]
    comparison_path = ABLATION_DIR / "fixed_70_15_15_ablation_summary.xlsx"
    comparison_df.to_excel(comparison_path, index=False)
    display(comparison_df)
    print(f"Saved comparison workbook: {comparison_path}")
else:
    print("No completed experiments found yet.")


##five-fold CV and significance testing

In [ ]:
# Five-fold CV for the main progressive models.
CV_EXPERIMENTS = [
    "cnn_bilstm",
    "cnn_bilstm_attention",
    "cnn_bilstm_attention_metadata",
    "cnn_bilstm_attention_metadata_adversarial",
]
cv_results, cv_summary = run_cross_validation(records, CV_EXPERIMENTS)
significance_results = run_paired_significance_tests(
    cv_results,
    reference_experiment="cnn_bilstm_attention_metadata_adversarial",
)


##LOSO evaluation

In [ ]:
# OPTIONAL — RUN THE TWO IMPORTANT LOSO COMPARISONS SEPARATELY
loso_non_adversarial = run_loso(records, experiment_name="cnn_bilstm_attention_metadata",)
loso_adversarial = run_loso(records, experiment_name="cnn_bilstm_attention_metadata_adversarial",)
